In [1]:
# Cài đặt PySpark
%pip install pyspark

In [2]:
# Import các thư viện cần thiết
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
import os

# Khởi tạo Spark Context
conf = SparkConf().setAppName("MovieRatingsAnalysis").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf=conf)
spark = SparkSession.builder.appName("MovieRatingsAnalysis").getOrCreate()

print("Spark Context đã được khởi tạo thành công!")

Spark Context đã được khởi tạo thành công!


In [3]:
# Đọc dữ liệu từ các file
import os

# Check if running in Google Colab
if 'COLAB_GPU' in os.environ or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    data_path = "/content/"
else:
    data_path = "data/" # For local environment

# Đọc file ratings_1.txt và ratings_2.txt
ratings_1_rdd = sc.textFile(data_path + "ratings_1.txt")
ratings_2_rdd = sc.textFile(data_path + "ratings_2.txt")

# Đọc file users.txt và occupation.txt
users_rdd = sc.textFile(data_path + "users.txt")
occupation_rdd = sc.textFile(data_path + "occupation.txt")

print(f"Số lượng rating từ file 1: {ratings_1_rdd.count()}")
print(f"Số lượng rating từ file 2: {ratings_2_rdd.count()}")
print(f"Số lượng user: {users_rdd.count()}")
print(f"Số lượng occupation: {occupation_rdd.count()}")

print("\nDữ liệu ratings_1.txt (5 dòng đầu):")
for line in ratings_1_rdd.take(5):
    print(line)

print("\nDữ liệu users.txt (5 dòng đầu):")
for line in users_rdd.take(5):
    print(line)

print("\nDữ liệu occupation.txt (5 dòng đầu):")
for line in occupation_rdd.take(5):
    print(line)

Số lượng rating từ file 1: 84
Số lượng rating từ file 2: 100
Số lượng user: 50
Số lượng occupation: 15

Dữ liệu ratings_1.txt (5 dòng đầu):
7,1020,4.5,1577836800
23,1015,3.5,1577923200
45,1030,4.0,1578009600
12,1047,3.0,1578096000
38,1012,4.5,1578182400

Dữ liệu users.txt (5 dòng đầu):
1,M,28,3,12345
2,F,35,7,23456
3,M,42,2,34567
4,F,19,10,45678
5,M,31,1,56789

Dữ liệu occupation.txt (5 dòng đầu):
1,Programmer
2,Doctor
3,Engineer
4,Teacher
5,Lawyer


In [4]:
# Xử lý dữ liệu occupation
# Parse occupation.txt: ID, Occupation
def parse_occupation(line):
    parts = line.split(',', 1)  # Tách thành 2 phần: ID, Occupation
    occupation_id = int(parts[0])
    occupation_name = parts[1] if len(parts) > 1 else f"Unknown {occupation_id}"
    return (occupation_id, occupation_name)

occupations_parsed = occupation_rdd.map(parse_occupation)
print("Occupations parsed (5 records):")
for occ in occupations_parsed.take(5):
    print(f"OccupationID: {occ[0]}, Occupation: {occ[1]}")

# Tạo dictionary để tra cứu tên occupation theo ID
occupations_dict = occupations_parsed.collectAsMap()
print(f"\nTổng số occupation trong dictionary: {len(occupations_dict)}")

# Xử lý dữ liệu users
# Parse users.txt: UserID, Gender, Age, Occupation, Zip-code
def parse_user_occupation(line):
    parts = line.split(',')
    user_id = int(parts[0])
    gender = parts[1]
    age = int(parts[2])
    occupation_id = int(parts[3])  # ID occupation
    zipcode = parts[4]
    return (user_id, occupation_id)

users_parsed = users_rdd.map(parse_user_occupation)
print("\nUsers parsed with occupation (5 records):")
for user in users_parsed.take(5):
    user_id, occ_id = user
    occ_name = occupations_dict.get(occ_id, f"Unknown {occ_id}")
    print(f"UserID: {user_id}, OccupationID: {occ_id}, Occupation: {occ_name}")

# Tạo dictionary để tra cứu occupation theo UserID
users_occupation_dict = users_parsed.collectAsMap()
print(f"\nTổng số user trong dictionary: {len(users_occupation_dict)}")

Occupations parsed (5 records):
OccupationID: 1, Occupation: Programmer
OccupationID: 2, Occupation: Doctor
OccupationID: 3, Occupation: Engineer
OccupationID: 4, Occupation: Teacher
OccupationID: 5, Occupation: Lawyer

Tổng số occupation trong dictionary: 15

Users parsed with occupation (5 records):
UserID: 1, OccupationID: 3, Occupation: Engineer
UserID: 2, OccupationID: 7, Occupation: Manager
UserID: 3, OccupationID: 2, Occupation: Doctor
UserID: 4, OccupationID: 10, Occupation: Accountant
UserID: 5, OccupationID: 1, Occupation: Programmer

Tổng số user trong dictionary: 50


In [5]:
# Xử lý dữ liệu ratings
# Parse ratings: UserID, MovieID, Rating, Timestamp
def parse_rating_with_user(line):
    parts = line.split(',')
    user_id = int(parts[0])
    movie_id = int(parts[1])
    rating = float(parts[2])
    timestamp = int(parts[3])
    return (user_id, rating)  # (user_id, rating) để join với users

# Parse cả 2 file ratings
ratings_1_parsed = ratings_1_rdd.map(parse_rating_with_user)
ratings_2_parsed = ratings_2_rdd.map(parse_rating_with_user)

print("Ratings 1 parsed (5 records):")
for rating in ratings_1_parsed.take(5):
    print(f"UserID: {rating[0]}, Rating: {rating[1]}")

print("\nRatings 2 parsed (5 records):")
for rating in ratings_2_parsed.take(5):
    print(f"UserID: {rating[0]}, Rating: {rating[1]}")

# Gộp 2 RDD ratings lại
all_ratings = ratings_1_parsed.union(ratings_2_parsed)
print(f"\nTổng số ratings từ cả 2 file: {all_ratings.count()}")

Ratings 1 parsed (5 records):
UserID: 7, Rating: 4.5
UserID: 23, Rating: 3.5
UserID: 45, Rating: 4.0
UserID: 12, Rating: 3.0
UserID: 38, Rating: 4.5

Ratings 2 parsed (5 records):
UserID: 12, Rating: 3.5
UserID: 34, Rating: 4.0
UserID: 27, Rating: 4.5
UserID: 8, Rating: 3.0
UserID: 19, Rating: 4.0

Tổng số ratings từ cả 2 file: 184


In [6]:
# Tính điểm trung bình và tổng số lượt đánh giá cho từng occupation

# Thêm occupation vào ratings
# all_ratings có format (user_id, rating)
# users_occupation_dict có format {user_id: occupation_id}

def add_occupation_to_rating(record):
    user_id, rating = record
    occupation_id = users_occupation_dict.get(user_id, -1)  # -1 nếu không tìm thấy
    occupation_name = occupations_dict.get(occupation_id, "Unknown")
    return (occupation_name, rating)

ratings_with_occupation = all_ratings.map(add_occupation_to_rating)

print("Ratings with occupation (5 records):")
for record in ratings_with_occupation.take(5):
    print(f"Occupation: {record[0]}, Rating: {record[1]}")

# Lọc bỏ những rating có occupation "Unknown"
valid_ratings = ratings_with_occupation.filter(lambda x: x[0] != "Unknown")

print(f"\nTổng số occupation-rating pairs hợp lệ: {valid_ratings.count()}")

# Tính tổng rating và số lượng rating cho mỗi occupation
# (occupation, rating) -> (occupation, (rating, 1))
occupation_ratings_with_count = valid_ratings.map(lambda x: (x[0], (x[1], 1)))

# Reduce theo key để tính tổng rating và tổng số lượng rating cho mỗi occupation
# (occupation, (sum_ratings, total_count))
occupation_stats = occupation_ratings_with_count.reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))

print(f"\nTổng số occupation có rating: {occupation_stats.count()}")
print("Occupation stats (5 records):")
for stat in occupation_stats.take(5):
    print(f"Occupation: {stat[0]}, Sum: {stat[1][0]}, Count: {stat[1][1]}")

Ratings with occupation (5 records):
Occupation: Designer, Rating: 4.5
Occupation: Consultant, Rating: 3.5
Occupation: Designer, Rating: 4.0
Occupation: Nurse, Rating: 3.0
Occupation: Journalist, Rating: 4.5

Tổng số occupation-rating pairs hợp lệ: 184

Tổng số occupation có rating: 14
Occupation stats (5 records):
Occupation: Consultant, Sum: 54.0, Count: 14
Occupation: Nurse, Sum: 42.5, Count: 11
Occupation: Lawyer, Sum: 62.0, Count: 17
Occupation: Manager, Sum: 55.5, Count: 16
Occupation: Student, Sum: 32.0, Count: 8


In [7]:
# Tính điểm trung bình cho từng occupation và hiển thị kết quả
def calculate_occupation_average(record):
    occupation, (sum_ratings, count) = record
    average_rating = sum_ratings / count
    return (occupation, (average_rating, count))

occupation_results = occupation_stats.map(calculate_occupation_average)

# Hiển thị tất cả các occupation theo định dạng yêu cầu
all_occupations = occupation_results.collect()

for occupation, (avg_rating, count) in all_occupations:
    print(f"{occupation} - AverageRating: {avg_rating:.2f} (TotalRatings: {count})")

Consultant - AverageRating: 3.86 (TotalRatings: 14)
Nurse - AverageRating: 3.86 (TotalRatings: 11)
Lawyer - AverageRating: 3.65 (TotalRatings: 17)
Manager - AverageRating: 3.47 (TotalRatings: 16)
Student - AverageRating: 4.00 (TotalRatings: 8)
Salesperson - AverageRating: 3.65 (TotalRatings: 17)
Engineer - AverageRating: 3.56 (TotalRatings: 18)
Designer - AverageRating: 4.00 (TotalRatings: 13)
Doctor - AverageRating: 3.69 (TotalRatings: 21)
Journalist - AverageRating: 3.85 (TotalRatings: 17)
Artist - AverageRating: 3.73 (TotalRatings: 11)
Programmer - AverageRating: 4.25 (TotalRatings: 10)
Accountant - AverageRating: 3.58 (TotalRatings: 6)
Teacher - AverageRating: 3.70 (TotalRatings: 5)


In [8]:
# Dọn dẹp tài nguyên
sc.stop()
spark.stop()
print("Đã dừng Spark Context và Spark Session.")

Đã dừng Spark Context và Spark Session.
